## Biais de positionnement A/B

In [9]:
import pandas as pd
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

In [4]:
df = pd.read_json("../data/comparia-reactions/reactions.jsonl", lines=True)

print(df.columns.tolist())
print(df.head(3))


['id', 'timestamp', 'model_a_name', 'model_b_name', 'refers_to_model', 'msg_index', 'opening_msg', 'conversation_a', 'conversation_b', 'model_pos', 'conv_turns', 'conversation_pair_id', 'conv_a_id', 'conv_b_id', 'refers_to_conv_id', 'session_hash', 'visitor_id', 'response_content', 'question_content', 'liked', 'disliked', 'comment', 'useful', 'creative', 'complete', 'clear_formatting', 'incorrect', 'superficial', 'instructions_not_followed', 'model_pair_name', 'msg_rank', 'question_id', 'system_prompt']
       id               timestamp  model_a_name          model_b_name  \
0  202099 2025-04-23 19:32:46.687   gemma-3-12b             command-a   
1  176467 2025-03-25 13:41:21.988  llama-3.1-8b  claude-3-5-sonnet-v2   
2  231166 2025-06-03 15:38:11.580  gpt-4.1-nano         llama-4-scout   

  refers_to_model  msg_index  \
0       command-a          1   
1    llama-3.1-8b          1   
2   llama-4-scout          1   

                                         opening_msg  \
0  Quelle est

In [10]:
def test_position(subset, label):
    ct = pd.crosstab(subset["model_pos"], subset["liked"])
    chi2, p, _, _ = chi2_contingency(ct)
    print(f"\n{label}")
    print(ct)
    print(f"chi2={chi2:.3f}  p={p:.4f}  {'significatif' if p < 0.05 else 'non significatif'}")

test_position(df, "GLOBAL")

for col in ["creative", "useful", "incorrect"]:
    test_position(df[df[col] == 1], f"Qualificatif : {col}")


GLOBAL
liked      False  True 
model_pos              
a          15213  31474
b          15075  32532
chi2=9.100  p=0.0026  significatif

Qualificatif : creative
liked      False  True 
model_pos              
a              6   3057
b              3   3288
chi2=0.601  p=0.4381  non significatif

Qualificatif : useful
liked      False  True 
model_pos              
a              9  10767
b             11  11589
chi2=0.003  p=0.9530  non significatif

Qualificatif : incorrect
liked      False
model_pos       
a           5115
b           5125
chi2=0.000  p=1.0000  non significatif


chi2_contingency est une fonction qui permet de voir la relation entre 2 groupes catégoriels. C'est une mesure statistique. avec cela est donné la p-value, cette mesure comparable à un seuil permet de dire s'il y a une différence statistque signifiquante entre les 2 groupes.

Dans notre cas nous comparons la position de la réponse du modèle (droite ou gauche dans la conversation) et le taux de réaction positive sachant que les 3 réactions possibles sont "creative", "useful" et "incorrect".

Pour le cas global nous avons : 
Position A : 31474 / (15213 + 31474) = 67.5% de likes
Position B : 32532 / (15075 + 32532) = 68.3% de likes
Donc il n'y a presque pas de différence pour les "like" mis.

Pour les "like" mis avec le qualificatif "creative" ou "useful" la position n'influence pas, de même pour le qualificatif "incorrect".

Cependant nous remarquons un autre point. Les réponses dites créative, utile sont presque toujours "liked" (inversement pour incorrecte). 

### Conclusion pour le biais de position

Le biais de position est presque inexistant or il se pourrait que les participants n'utilisent "creative" que quand ils aiment déjà la réponse et ne l'utilisent pas comme critère indépendant. Les participants ne dissocient pas créativité et appréciation générale.
